# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

from agents.deals import ScrapedDeal
from agents.deals_common import DealSelection

import nest_asyncio

In [2]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

nest_asyncio.apply()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<?, ?it/s]


In [4]:
len(deals)

90

In [5]:
deals[44].describe()

"Title: Walmart Can't Miss Clearance: Up to 75% off + free shipping w/ $35\nDetails: Find clearance savings on clothing, home improvement, furniture, sports and outdoors, automotive, and more. Shipping adds $7, or get free shipping over $50. Shop Now at Walmart\nFeatures: \nURL: https://www.dealnews.com/Walmart-Cant-Miss-Clearance-Up-to-75-off-free-shipping-w-35/21739910.html?iref=rss-c196"

In [6]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [7]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [8]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Motorola G 5G 128GB Android Phone for Straight Talk for $40 + free shipping
Details: It's the same price you'd pay for the same phone with half the storage elsewhere (Walmart charges $40 for the 64GB model; this one is the 128GB model). Buy Now at eBay
Features: 6.5" display Snapdragon 4 Gen. 1 5G processor 128GB storage; 4GB RAM 5,000mAh battery Model: STMTXT2417DC

In [9]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [10]:
result = get_recommendations()

In [11]:
print(type(result))
len(result.deals)

<class 'agents.deals_common.DealSelection'>


5

In [12]:
result.deals[1]

Deal(product_description='This Samsung QE1D QN85QE1DAFXZA 85-inch QLED Smart TV delivers stunning 4K resolution and vibrant colors for an exceptional viewing experience. As one of the largest models available, it’s designed to be the centerpiece of your entertainment setup. Equipped with smart technology, it provides access to popular streaming services and features for enhanced usability. The impressive size combined with high-quality display technology makes this TV a compelling choice for movie enthusiasts and gamers alike.', price=1100.0, url='https://www.dealnews.com/products/Samsung/Samsung-QE1-D-QN85-QE1-DAFXZA-85-QLED-4-K-Smart-TV/490380.html?iref=rss-c142')

## ScannerAgent is using 'gpt-4o-mini'

In [13]:
from agents.scanner_agent import ScannerAgent

In [14]:
agent = ScannerAgent(show_progress=True)
result = agent.scan()

100%|████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<?, ?it/s]


In [15]:
result.deals

[Deal(product_description='The Samsung QE1D QN85QE1DAFXZA is an impressive 85-inch QLED 4K smart television that combines top-tier brightness with stunning color accuracy, making it perfect for movie nights or gaming sessions. With its advanced technology, the TV provides sharp images and vibrant colors, enhancing the viewing experience. Features include Quantum HDR and ownership of Samsung smart TV capabilities that allow for easy access to streaming services and apps, offering both convenience and quality viewing.', price=1100.0, url='https://www.dealnews.com/products/Samsung/Samsung-QE1-D-QN85-QE1-DAFXZA-85-QLED-4-K-Smart-TV/490380.html?iref=rss-c142'),
 Deal(product_description='Unlock the potential of seamless communication and endless entertainment with the Unlocked Google Pixel Fold 256GB Android Smartphone. This premium device features a versatile folding design, initiated for multitasking and enhanced usability. With its exceptional camera system and stock Android experience, 

In [16]:
print(result.deals[0])

product_description='The Samsung QE1D QN85QE1DAFXZA is an impressive 85-inch QLED 4K smart television that combines top-tier brightness with stunning color accuracy, making it perfect for movie nights or gaming sessions. With its advanced technology, the TV provides sharp images and vibrant colors, enhancing the viewing experience. Features include Quantum HDR and ownership of Samsung smart TV capabilities that allow for easy access to streaming services and apps, offering both convenience and quality viewing.' price=1100.0 url='https://www.dealnews.com/products/Samsung/Samsung-QE1-D-QN85-QE1-DAFXZA-85-QLED-4-K-Smart-TV/490380.html?iref=rss-c142'


In [17]:
print(type(result))

<class 'agents.deals_common.DealSelection'>
